# 07. Generation - LLM 답변 생성 (시나리오 B)

## 목표
- 검색된 컨텍스트를 기반으로 LLM이 정확한 답변을 생성
- 질문에서 메타데이터 필터를 자동 추출하여 검색 정확도 향상
- 대화 맥락 유지 (후속 질문 대응)
- 문서에 없는 내용은 모른다고 답변

## 모델
- **gpt-5-mini** (시나리오 B 베이스라인)
- 허용 모델: gpt-5-mini, gpt-5-nano, text-embedding-3-small

> **Input**: `artifacts/chroma_db/` + `data/processed/chunks.parquet`  
> **Output**: RAG 답변 (in-memory)  
> **Prev**: [06_retrieval.ipynb](06_retrieval.ipynb) | **Next**: [08_evaluation.ipynb](08_evaluation.ipynb)


In [1]:
import pandas as pd
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
openai_client = OpenAI()

In [2]:
ARTIFACTS_DIR = Path("../../artifacts")
CHROMA_DIR = ARTIFACTS_DIR / "chroma_db"
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-5-mini"
COLLECTION_NAME = "rfp_chunks_v1"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION_NAME)
print(f"컬렉션: {collection.count():,}개 청크, 모델: {LLM_MODEL}")

컬렉션: 13,951개 청크, 모델: gpt-5-mini


## 1. 검색 함수 (06번에서 가져옴)

In [3]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(input=query, model=EMBEDDING_MODEL)
    return response.data[0].embedding


def retrieve(query: str, k: int = 5, where: dict = None, where_document: dict = None) -> list[dict]:
    """벡터 DB에서 유사 청크를 검색한다."""
    query_embedding = embed_query(query)
    kwargs = {"query_embeddings": [query_embedding], "n_results": k}
    if where:
        kwargs["where"] = where
    if where_document:
        kwargs["where_document"] = where_document
    results = collection.query(**kwargs)
    chunks = []
    for i in range(len(results['ids'][0])):
        chunks.append({
            'score': round(1 - results['distances'][0][i], 4),
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
        })
    return chunks

## 2. 질문 분석 — 메타데이터 필터 자동 추출

사용자 질문에서 발주기관, 사업명 등을 감지하여 검색 필터를 자동 생성합니다.
06번에서 확인한 것처럼 메타 필터를 걸면 정확도가 크게 올라갑니다.

In [4]:
# 발주기관 목록 로딩 (퍼지 매칭용)
all_meta = collection.get(limit=1, include=['metadatas'])  # 전체 메타 가져오기 비효율 -> 별도 로딩
chunks_df = pd.read_parquet(Path("../../data/processed/chunks.parquet"))
AGENCY_LIST = chunks_df['발주 기관'].unique().tolist()
PROJECT_LIST = chunks_df['사업명'].unique().tolist()
print(f"발주기관: {len(AGENCY_LIST)}개, 사업: {len(PROJECT_LIST)}개")

발주기관: 87개, 사업: 99개


In [5]:
# 메타데이터 필터 매핑 테이블
DOMAIN_KEYWORDS = {
    '교육/학습': ['교육', '학습', '이러닝', '학사', 'LMS', '연수', '대학'],
    '안전/재난': ['안전', '재난', '방재', '관제', '선량', '방사선'],
    '웹/포털': ['홈페이지', '포털', '웹', '온라인서비스'],
    '경영/행정': ['ERP', '그룹웨어', '경영', '인사', '회계', '오피스'],
    '공간정보/GIS': ['GIS', '지도', '공간정보', '측량', '수문', '관개'],
    '의료/바이오': ['의료', '건강', '바이오', '병원', '보험'],
    'ISP/컨설팅': ['ISP', '전략', '컨설팅', '타당성'],
    'AI/데이터': ['AI', '인공지능', '빅데이터', '데이터분석', '머신러닝'],
    '교통/물류': ['버스', '교통', 'BIS', 'ITS', '물류'],
    '농축수산': ['축산', '농업', '수산', '어촌'],
    '문화/콘텐츠': ['문화', '예술', '박물관', '아카이브', '영화'],
    '복지/사회서비스': ['복지', '돌봄', '사회보험', '서민금융'],
}

AGENCY_TYPE_KEYWORDS = {
    '대학교': ['대학', '학교'],
    '공기업/준정부기관': ['공사', '공단', '진흥원'],
    '지방자치단체': ['시청', '군청', '구청', '지자체'],
    '연구기관': ['연구원', '연구소'],
}

TECH_KEYWORDS = {
    'AI': ['AI', '인공지능', '머신러닝', '딥러닝'],
    '클라우드': ['클라우드', 'cloud', 'SaaS'],
    'IoT': ['IoT', '센서', 'SCADA'],
    'GIS': ['GIS', '지도', '공간정보'],
    '모바일': ['모바일앱', '모바일 앱'],
}


def extract_filters(query: str, chat_history: list = None) -> dict:
    """질문에서 메타데이터 필터를 추출한다.
    
    우선순위:
    1. 발주기관명 직접 매칭 (가장 강력)
    2. 사업도메인 키워드 매칭
    3. 기관유형 키워드 매칭
    4. 대화 맥락에서 이전 필터 유지
    """
    # 필터 해제 키워드
    release_keywords = ['다른 기관', '다른 사업', '그 외', '외에', '이외', '말고']
    if any(kw in query for kw in release_keywords):
        return None
    
    where = {}
    
    # 1) 발주기관명 직접 매칭
    matched_agencies = []
    for agency in AGENCY_LIST:
        short = agency.replace('(주)', '').replace('㈜', '').strip()
        for part in [short, short[:6], short[:4]]:
            if len(part) >= 3 and part in query:
                if agency not in matched_agencies:
                    matched_agencies.append(agency)
                break
    
    if len(matched_agencies) >= 2:
        return None  # 다문서 비교
    if len(matched_agencies) == 1:
        where['발주기관'] = matched_agencies[0]
        return where
    
    # 2) 사업도메인 키워드 매칭
    for domain, keywords in DOMAIN_KEYWORDS.items():
        if any(kw in query for kw in keywords):
            where['사업도메인'] = domain
            break
    
    # 3) 기관유형 키워드 매칭
    if '사업도메인' not in where:
        for atype, keywords in AGENCY_TYPE_KEYWORDS.items():
            if any(kw in query for kw in keywords):
                where['기관유형'] = atype
                break
    
    if where:
        return where
    
    # 4) 대화 맥락에서 이전 필터 유지
    if chat_history:
        for msg in reversed(chat_history):
            if msg.get('filter'):
                return msg['filter']
    
    return None


# 테스트
test_queries = [
    "국민연금공단이 발주한 이러닝시스템 사업 요구사항",
    "고려대학교 차세대 포털이랑 광주과학기술원 학사 시스템 비교",
    "교육 관련 사업 찾아줘",
    "AI 기반 예측 시스템 요구사항",
    "대학교에서 발주한 사업 목록",
    "안전 관련 시스템 사업이 있어?",
    "사업 예산이 얼마야",
]
for q in test_queries:
    f = extract_filters(q)
    print(f"  {q[:40]:40s} -> {f}")


  국민연금공단이 발주한 이러닝시스템 사업 요구사항               -> {'발주기관': '국민연금공단'}
  고려대학교 차세대 포털이랑 광주과학기술원 학사 시스템 비교         -> None
  교육 관련 사업 찾아줘                             -> {'사업도메인': '교육/학습'}
  AI 기반 예측 시스템 요구사항                        -> {'사업도메인': 'AI/데이터'}
  대학교에서 발주한 사업 목록                          -> {'사업도메인': '교육/학습'}
  안전 관련 시스템 사업이 있어?                        -> {'사업도메인': '안전/재난'}
  사업 예산이 얼마야                               -> None


## 3. 프롬프트 정의

In [6]:
SYSTEM_PROMPT = """당신은 공공입찰 RFP(제안요청서) 분석 전문가입니다.
사용자의 질문에 대해 제공된 RFP 문서 컨텍스트를 기반으로 정확하게 답변합니다.

## 역할
- 입찰메이트 컨설팅 스타트업의 사내 RAG 시스템
- 컨설턴트가 RFP 핵심 정보를 빠르게 파악하도록 지원

## 답변 원칙
1. 반드시 제공된 컨텍스트에 있는 내용만 답변하세요.
2. 컨텍스트에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다"라고 명확히 답변하세요. 추측하지 마세요.
3. 부분적으로만 확인되는 경우 "문서에서 확인된 내용은 다음과 같으며, [X]에 대한 정보는 포함되어 있지 않습니다"로 구분하세요.

## 답변 형식
- 출처를 [사업명 | 발주기관] 형태로 답변 끝에 명시하세요.
- 표 형태의 정보는 마크다운 표 구조를 유지하세요.
- 요구사항 정리 시 구분(기능/성능/보안 등)과 고유번호를 포함하세요.
- 핵심을 먼저 말하고 세부사항을 이어서 설명하세요.

## 질문 유형별 대응
- **단일 문서 질문**: 해당 문서의 관련 섹션을 구조적으로 정리
- **다문서 비교**: 비교 항목(사업개요, 예산, 기간, 주요 요구사항)을 하나의 표로 나란히 비교. 문서별로 별도의 표를 만들지 말고, 항목을 행으로, 문서를 열로 배치하여 간결하게 작성
- **후속 질문**: 이전 대화 맥락을 참고하되, 현재 컨텍스트 기반으로 답변
- **존재하지 않는 정보**: "찾을 수 없습니다"와 함께 관련된 유사 정보가 있으면 안내"""


def build_context(chunks: list[dict], max_chars: int = 8000) -> str:
    """검색된 청크를 LLM 컨텍스트로 조합한다."""
    parts = []
    total = 0
    for c in chunks:
        source = f"[출처: {c['metadata']['사업명']} | {c['metadata']['발주기관']}]"
        chunk_text = f"{source}\n{c['text']}"
        if total + len(chunk_text) > max_chars:
            break
        parts.append(chunk_text)
        total += len(chunk_text)
    return "\n\n---\n\n".join(parts)


print(f"시스템 프롬프트: {len(SYSTEM_PROMPT)}자")


시스템 프롬프트: 732자


## 4. RAG 파이프라인 통합

In [7]:
def rag_query(
    query: str,
    chat_history: list = None,
    k: int = 5,
    model: str = LLM_MODEL,
) -> dict:
    """질문 -> 필터 추출 -> 검색 -> 답변 생성 전체 파이프라인."""
    start = time.time()
    
    # 1) 필터 추출
    where = extract_filters(query, chat_history)
    
    # 2) 검색
    chunks = retrieve(query, k=k, where=where)
    context = build_context(chunks)
    
    # 3) 메시지 구성
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    # 대화 히스토리 추가 (최근 4턴)
    if chat_history:
        for msg in chat_history[-4:]:
            messages.append({"role": "user", "content": msg["user"]})
            messages.append({"role": "assistant", "content": msg["assistant"]})
    
    user_message = f"""다음 RFP 문서 내용을 참고하여 질문에 답변해주세요.

## 참고 문서
{context}

## 질문
{query}"""
    messages.append({"role": "user", "content": user_message})
    
    # 4) LLM 호출
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=8000,
    )
    
    answer = response.choices[0].message.content or '(응답 없음)'
    elapsed = time.time() - start
    
    return {
        "query": query,
        "answer": answer,
        "filter": where,
        "chunks_used": len(chunks),
        "context_chars": len(context),
        "elapsed_sec": round(elapsed, 1),
        "model": model,
        "tokens": {
            "prompt": response.usage.prompt_tokens,
            "completion": response.usage.completion_tokens,
            "total": response.usage.total_tokens,
        },
    }


def display_answer(result: dict):
    """답변 결과를 정리하여 출력한다."""
    print(f"Q: {result['query']}")
    print(f"필터: {result['filter']}")
    print(f"검색: {result['chunks_used']}개 청크, {result['context_chars']:,}자 컨텍스트")
    print(f"토큰: {result['tokens']['prompt']}+{result['tokens']['completion']}={result['tokens']['total']}")
    print(f"시간: {result['elapsed_sec']}초")
    print(f"\nA:\n{result['answer']}")
    print("=" * 60)


## 5. 단일 문서 질문 테스트 (유형 A)

In [8]:
display_answer(rag_query(
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘"
))

Q: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 4,865자 컨텍스트
토큰: 2969+2998=5967
시간: 49.2초

A:
핵심 요약
- 요구사항은 크게 교육운영, 이러닝시스템 운영(시스템기능·학사관리), 콘텐츠 개발·유지관리, 개인정보·정보보안, 프로젝트관리(계약·인수인계 등)로 구성되어 있습니다. 각 요구사항은 문서에 고유번호(FUR/SFR/DER/... 등)로 정리되어 있으나, 개별 항목의 세부 기능·성능지표·산출물·평가기준 등 구체명세는 제공된 문서에 포함되어 있지 않습니다.  
- 추가로 실물 교재 지급 항목은 반드시 배송료 포함 가격 제시, 산출내역서 작성 시 제안 안내(p20~26) 참고, 부서별 필수교육 운영단가는 직무교육 운영단가에 준함 등의 운영·계약 관련 지침이 명시되어 있습니다.  
- 인수인계 시 공단 보유 프로그램·데이터 이전 방안 제시 및 정보보안·개인정보보호 규정 준수·보안 관리 방안 제시가 요구됩니다.

요구사항 정리 (분류 / 고유번호 / 요구사항 명칭 / 문서에 명시된 내용 / 문서에 없는(또는 상세 미기재) 내용)
| 분류 | 고유번호 | 요구사항 명칭 | 문서에 명시된 내용(확인된 내용) | 상세(문서에서 찾을 수 없음) |
| --- | --- | --- | --- | --- |
| 교육운영(기능) | FUR-001 | 국민연금공단 역량 모델 기반의 TRM 제시 | 요구목록에 명시 | TRM의 구성요소, 산출물, 형식, 평가기준 등 구체사항 없음 |
| 교육운영(기능) | FUR-002 | 교육운영 방법 | 요구목록에 명시 | 운영 방식(오프라인/온라인 비중 등), 일정, 운영절차 미기재 |
| 교육운영(기능) | FUR-003 | 교육 예상인원 | 요구목록에 명시 | 예상인원 상세수, 인원별 처리방안 등 미기재 |
| 교육운영(기능) | FUR-004 | 외부콘텐츠 제시 기준 | 요구목록에 명시 | 제시 기준의 상세 항목(품질기준,

In [9]:
display_answer(rag_query(
    "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘"
))

Q: 한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘
필터: {'사업도메인': '안전/재난'}
검색: 5개 청크, 3,771자 컨텍스트
토큰: 2697+983=3680
시간: 15.4초

A:
핵심 요약: 규제요건 준수와 업무생산성 향상, 그리고 시스템 신뢰성 확보를 위해 한국원자력연구원 정상운전 시 선량평가시스템(K-RADAC)을 고도화하려는 사업입니다.  
세부 목적(문서에서 확인된 내용):

1) 규제요건 준수 (목적 1)  
   - 원안위 고시 제2019-10호 「방사선방호 등에 관한 기준」 제16조(환경상의 위해방지)에 따른 제한구역 경계에서의 연간선량 준수 여부를 확인할 수 있는 체계 구축 필요.  
   - ICRP60 기반 평가장기 개선 필요.  
   - 액체유출물에 의한 주민피폭선량평가 수행 기능 추가 필요.

2) 업무생산성 향상 (목적 2)  
   - UI 개선을 통한 업무개선 및 신속한 의사결정 환경 구축 필요.  
   - 출력 기능 고도화를 통한 생산성 향상 필요.

3) 추진 목표 및 기대효과 (목적 3)  
   - 추진목표: 정상운전 시 주민선량평가시스템(K-RADAC) 고도화.  
   - 기대효과: 정상운전 시 선량평가 관련 규제 수요 대응, 데이터 신뢰성 확보 및 생산성 향상.

출처: [한국원자력연구원 선량평가시스템 고도화 | 한국원자력연구원]


## 6. 후속 질문 테스트 (유형 C)

대화 맥락을 유지하여 후속 질문에 답변합니다.

In [10]:
# 첫 질문
r1 = rag_query("국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘")
display_answer(r1)

# 대화 히스토리 구성
history = [{
    "user": r1["query"],
    "assistant": r1["answer"],
    "filter": r1["filter"],
}]

Q: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 4,865자 컨텍스트
토큰: 2969+2942=5911
시간: 43.7초

A:
핵심 요약
- 요구사항은 교육운영, 이러닝시스템 운영, 콘텐츠 개발·관리, 유지관리·프로젝트지원, 개인정보·정보보안, 저작권 등 제약, 프로젝트관리 등으로 분류되어 있으며 각 항목에 고유번호(FUR, QUR, SFR, LMR, DER, MAR, PSR, SER, COR, PMR 등)가 지정되어 있습니다. 문서에는 요구사항 명칭(목록)이 제시되어 있으나, 개별 요구사항의 상세 기술(세부요건)은 본문의 해당 섹션에서 확인해야 합니다. 문서에서 확인된 내용과, 문서에 포함되지 않은 세부 정보는 아래에 구분하여 정리합니다.

요구사항 목록 (문서에 명시된 고유번호·명칭)
| 구분 | 고유번호 | 요구사항 명칭 | 문서상 확인된 내용(요약) |
| --- | ---: | --- | --- |
| 교육운영(기능) | FUR-001 | 국민연금공단 역량 모델 기반의 TRM 제시 | 요구사항 명칭만 제시되어 있음. 세부요건은 문서 해당 섹션에 별도 기술되어 있을 것으로 보임. 문서에서 세부내용은 포함되어 있지 않습니다. |
| 교육운영(기능) | FUR-002 | 교육운영 방법 | 명칭만 제시됨. 세부방법은 문서에 포함되어 있지 않습니다. |
| 교육운영(기능) | FUR-003 | 교육 예상인원 | 명칭만 제시됨. 단, 일부 교육별 인원(예: 북러닝 800명, 전화외국어 110명(2개월), 70명(3개월))은 문서에 별도로 표기되어 있음. |
| 교육운영(기능) | FUR-004 | 외부콘텐츠 제시 기준 | 명칭만 제시됨. 세부기준은 문서에 포함되어 있지 않습니다. |
| 교육운영(기능) | FUR-005 | 외부콘텐츠 기타 요청사항 | 명칭만 제시됨. 세부요청사항은 문서에 포함되어 있지 않습니다. |
| 교육운영(품질) | QUR-001 | 교육콘텐츠 평가 및 

In [11]:
# 후속 질문 — "콘텐츠 개발 관리" (맥락: 국민연금공단 이러닝)
r2 = rag_query("콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘", chat_history=history)
display_answer(r2)

Q: 콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 5,071자 컨텍스트
토큰: 5200+3626=8826
시간: 64.4초

A:
핵심 요약  
- 문서에 명시된 콘텐츠 개발·관리 요구사항은 크게 “개발 범위(DER-001)”, “개발 요건(DER-002)”, “검사·검수(DER-003)”, “관리 범위(MAR-001)”, “수행사항(MAR-002)” 등으로 구분됩니다.  
- 주요 핵심사항: 기존 공단 콘텐츠 수정 및 신규 60차시 개발, 부서별 필수교육 10차시 개발(DER-001); 동영상 중심(차시당 30~40분), 웹·모바일 연동(스마트러닝), 모듈화 설계, ‘나의 학습노트’ 연동 등(DER-002); 원본·완성본 및 PPT형태 다운로드 제공, 검수 시 즉시 시정 및 보고(DER-003, DER-002); 업데이트·법령 반영·즉시조치·1년 하자보수(부분)(MAR-002).  

요구사항 요약표 (고유번호 · 분류 · 요약)
| 고유번호 | 분류 | 요약(문서상 확인된 핵심 내용) |
| --- | --- | --- |
| DER-001 | 개발(범위) | 기 개발 콘텐츠 수정 + 신규 60차시 개발, 부서별 필수교육 10차시 개발 |
| DER-002 | 개발(요건) | 즉시 개발·컨설팅 지원·개발계획서 제출·개발안내 워크숍, 동영상 30~40분, 웹·모바일 연동(스마트러닝), 모듈화(차시당 1개 모듈), ‘나의 학습노트’ 연동, 디자인/이미지 3개 이상 제안, 프로토타입 → 공단승인 후 전체 개발, 공단 네트워크 환경 고려(부하 최소화) |
| DER-003 | 품질/검수 | 모든 개발물 검사·검수, 시정요구 즉시 반영·서면 결과보고, 원본·완성본 제공(인재개발부 1, 해당부서 1), PPT다운로드 기능 제공 |
| MAR-001 | 유지관리(관리범위) | 프로그램·콘텐츠 목록(나의학습노트, 역량진단 등), 현재 사용기술 스택 표기(MS-SQL2005, JAVA, JSP, Ac

In [12]:
# 후속 질문 — 다른 기관 탐색
history.append({"user": r2["query"], "assistant": r2["answer"], "filter": r2["filter"]})
r3 = rag_query("교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?", chat_history=history)
display_answer(r3)

Q: 교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?
필터: None
검색: 5개 청크, 3,505자 컨텍스트
토큰: 6844+1556=8400
시간: 24.0초

A:
핵심 요약  
예 — 제공된 문서들에는 국민연금공단 외에 다른 기관이 발주한 교육·학습 관련 사업(또는 교육계획·운영 요구를 포함한 사업)이 포함되어 있습니다. 아래 표에 문서에 명시된 사업명과 발주기관, 교육 관련 핵심내용을 정리합니다.

| 사업명 | 발주기관 | 교육/학습 관련 핵심내용(문서에 확인된 것) |
| --- | --- | --- |
| 인천일자리플랫폼 정보시스템 구축 ISP 수립용역 | 인천광역시 | 교육훈련계획 제출 요구(목적·대상·일정·장소·내용·강사 등), 발주기관 직원 대상 산출물 활용 교육 필요, 배포용 교재 및 소요경비는 사업자가 부담, 교육결과보고서 제출 등(교육계획서·교육결과보고서 산출물 명시). |
| 국가교육과정정보센터(NCIC) 시스템 운영 및 개선 | 한국교육과정평가원 | 사업서식에 ‘유사분야 사업수행 실적(최근 3년)’ 항목 포함(교육·시스템 운영 관련 실적 기재 양식 포함). (문서에 실적 기재 양식과 참고사항이 제시되어 있음.) |
| e나라도움 업무시스템 웹 접근성 컨설팅 | 한국재정정보원 | 운영자 교육 계획 수립 및 운영자 교육(방법·절차·횟수 등) 실시 요구, 교육 결과보고서 제출, 사후관리·가이드라인 제공 등(교육계획서·교육결과보고서 산출물 명시). |

추가 참고 및 한계  
- 위 내용은 제공된 문서 텍스트에 명시된 교육·학습 관련 항목만을 정리한 것입니다.  
- 제공된 문서들 외에 다른 발주기관의 교육 관련 사업은 문서 내에서 찾을 수 없습니다. (추가 문서가 있으면 더 찾아 정리해 드리겠습니다.)

출처: [인천일자리플랫폼 정보시스템 구축 ISP 수립용역 | 인천광역시], [국가교육과정정보센터(NCIC) 시스템 운영 및 개선 | 한국교육과정평가원], [e나라도움 업무시스템 웹 접근성 컨설팅 | 한국재정정보원]


## 7. 다문서 비교 테스트 (유형 B)

In [13]:
display_answer(rag_query(
    "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?",
    k=10,  # 두 문서를 모두 가져오기 위해 k 증가
))

Q: 고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?
필터: None
검색: 10개 청크, 8,022자 컨텍스트
토큰: 5016+2726=7742
시간: 39.6초

A:
핵심 요약
- 주요 차이: 사업 규모와 기간, 요구 범위 및 평가 방식에서 큰 차이가 있음. 고려대 사업은 대형(예산 112.7억, 24개월, 포털·통합 학사시스템 전면 구축·응용SW개발, 상세한 평가기준 및 보안·아키텍처 요구), 광주과기원(GIST) 사업은 소규모(예산 1.573억원, 계약일로부터 5개월, 학사조직 개편에 따른 기능개선 중심, 간단한 비용·지급조건).  
- 세부 비교표(항목을 행으로, 문서를 열로 배치):

| 항목 | 학사시스템 기능개선 사업 | 차세대 포털·학사 정보시스템 구축 사업 |
| --- | ---: | --- |
| 사업명 | 학사시스템 기능개선 사업 | 차세대 포털·학사 정보시스템 구축 사업 |
| 발주기관 | 광주과학기술원 | 고려대학교 |
| 사업기간 | 계약체결일로부터 5개월 이내 | 계약일로부터 24개월 이내 |
| 사업비(예산) | 157,300,000원 (VAT 포함) | 11,270,000,000원 (VAT 포함, 3년 분할 지급) |
| 비용내역(제공된 항목) | 설계·분석(업무분석자) 53,000,000원(총 11M/M), 개발·테스트(응용SW 개발자) 90,000,000원 등(소계 143,000,000) | 문서에 상세 품목별 예산 분해는 없음(총액 및 지급연도 비율 제시: 2024 ~30%, 2025 ~40%, 2026 ~30%) |
| 계약방법 | 공개경쟁입찰(협상에 의한 계약체결) | 제한경쟁입찰(협상에 의한 계약) |
| 대금지급방법 | 선금: 계약금의 50% 지급, 잔금: 최종검수 완료시 지급 | 문서에 상세 지급조건은 없고 3개년 분할 지급 방식 명시(무상유지보수 기간 포함) |
| 공동수급 | 공동수급 불가 | 문서에 해당 정보 없음 |
| 무상유지보수 | 문서에 해당 정보 

## 8. 무응답 테스트 (유형 D)

문서에 없는 내용에 대해 모른다고 답변하는지 확인합니다.

In [14]:
display_answer(rag_query(
    "국민연금공단 이러닝시스템 사업의 낙찰 업체가 어디야?"
))

Q: 국민연금공단 이러닝시스템 사업의 낙찰 업체가 어디야?
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 3,916자 컨텍스트
토큰: 2924+713=3637
시간: 12.1초

A:
핵심: 제공된 문서에서 낙찰업체(낙찰자) 정보를 찾을 수 없습니다.

세부: 첨부된 내용은 제안요청서 양식(입찰참가신청서, 입찰보증금 안내, 업체현황·프로젝트 일정 서식 등)으로, 낙찰자(낙찰업체) 명칭이나 낙찰 결과에 대한 정보는 포함되어 있지 않습니다. 문서에서 해당 정보를 찾을 수 없습니다.

[2024년 이러닝시스템 운영 용역 | 국민연금공단]


In [15]:
display_answer(rag_query(
    "2025년 서울시 AI 챗봇 구축 사업에 대해 알려줘"
))

Q: 2025년 서울시 AI 챗봇 구축 사업에 대해 알려줘
필터: {'발주기관': '2025 구미 아시아육상경기선수권대회 조직위원회'}
검색: 5개 청크, 5,160자 컨텍스트
토큰: 3323+2331=5654
시간: 31.1초

A:
제공된 문서에서 "2025년 서울시 AI 챗봇 구축 사업"에 대한 정보는 찾을 수 없습니다.

대신 제공된 문서(2025 구미아시아육상경기선수권대회 종합정보시스템 RFP)에서 소프트웨어·시스템 구축사업에 일반적으로 적용되는 요구사항 중 AI 챗봇 RFP에 참고될 수 있는 항목들은 다음과 같습니다. 핵심을 먼저 제시하고 세부를 이어갑니다.

핵심 요약
- 문서에는 품질관리, 성능, 데이터·DB 관리, SW사업정보 제출, 평가기준 등 소프트웨어 사업 전반의 요구사항이 상세히 제시되어 있음.  
- 직접적인 "서울시 AI 챗봇" 관련 내용은 제공문서에 포함되어 있지 않습니다.

세부 요구사항 (문서에서 확인된 내용)
| 요구사항 구분 | 고유번호 | 분류 | 주요 내용 요약 |
| --- | ---:| --- | --- |
| 품질관리 준수 | QUR-002 | 품질 요구사항 | - 품질관리 조직 및 운영절차 구체 제시·준수<br>- 품질/성능 문제 발생 시 부하테스트 및 분석결과 제시·개선<br>- 과업 범위 외 요인(운영서버 등) 영향 시 원인·해결방안 제시 및 협조<br>- DB구축 포함 시 사업완료(검수)시 데이터 품질진단 및 오류데이터 개선 실시 |
| 성능 일반 | (고유번호 미기재) | 성능 요구사항 | - 성능 고려한 개발방안 제시 및 안정적 운영 지원 방안 제시<br>- 개발 중 성능 모니터링으로 사전 문제 파악·조치<br>- 기존 장비 활용 및 기술지원·가이드 제공<br>- 사용자가 입력한 정보에서 발생하는 모든 오류 메시지를 입력 후 3초 이내 표시(중요정보 노출 금지) |
| 데이터·DB 관리 | (고유번호 미기재) | 데이터 요구사항 | - 외부 데이터 연계 시 데이터 무결성 보장<br>- 개발되거나 변경되는 

## 9. 모호한 질의 테스트 (유형 E)

In [16]:
display_answer(rag_query(
    "기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?"
))

Q: 기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?
필터: {'발주기관': '기초과학연구원'}
검색: 5개 청크, 5,117자 컨텍스트
토큰: 3164+749=3913
시간: 13.3초

A:
핵심: 제공된 문서 내에 AI 기반 예측(예: AI, 머신러닝, 예측유지관리, Predictive analytics) 요구사항은 없습니다. 문서에서 해당 정보를 찾을 수 없습니다.

세부사항(문서에서 확인된 관련 내용)
- 휴지기간 모니터링 업무(표 9): 알람 메시지 확인, 주요 센서 트렌드(온도·압력·유량·속도) 확인, 비정상 상황 감지, 제어로직 정상 작동 여부 확인 등 데이터·모니터링 관련 업무는 명시되어 있음.
- 문서 제출 항목: 극저온시스템 주요 수치 및 수치 기록을 계약자가 PC에 상시 저장하여 제출하도록 규정되어 있음(월간 기성 신청 시 제출 등).
- 제어시스템 개선 협조: 제어시스템(알람시스템 포함) 개선 및 보완을 위한 협조 항목이 있으나, 이 또한 AI 적용 요구를 명시하지 않음.

결론: 데이터 수집·모니터링 요구는 있으나, AI 기반 예측 기술의 적용 또는 요구에 대한 언급은 제공된 문서에 포함되어 있지 않습니다. [2025년도 중이온가속기용 극저온시스템 운전 용역 | 기초과학연구원]


In [17]:
# 모니터링 관련 후속 질문
r_cryo = rag_query("기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?")
history_cryo = [{"user": r_cryo["query"], "assistant": r_cryo["answer"], "filter": r_cryo["filter"]}]

display_answer(rag_query(
    "그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘",
    chat_history=history_cryo,
))

Q: 그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘
필터: {'발주기관': '기초과학연구원'}
검색: 5개 청크, 5,124자 컨텍스트
토큰: 4112+2926=7038
시간: 45.2초

A:
핵심 결론
- 예. 문서에 모니터링 업무에 대한 구체적 요청사항이 명시되어 있습니다. 휴지기간(비가동기)과 가동기간 모두에 대해 모니터링 항목·절차·기록·분석 의무가 규정되어 있습니다.  
- 참고: 문서에서는 AI 기반 예측(예: 머신러닝·예측진단) 관련 요구사항은 포함되어 있지 않습니다. 제공된 문서에서 해당 정보를 찾을 수 없습니다.

모니터링 업무 요약(핵심 먼저, 세부 후술)
- 목적: 휴지기간 동안 극저온시스템의 안정적 가동 보장(유지보수 전후 특이사항 확인 및 결과 확인 포함).  
- 수행수단: 연구원이 구축한 제어시스템(WinCC, CS‑studio 등) 또는 현장 장치 사용. 연구원이 기본 사용 교육을 제공함.  
- 주요 활동: 알람 확인, 주요 센서 트렌드 확인·정상치 숙지, 비정상 감지, 제어로직 정상성 확인, 연구원 요청 시 현장 일시/지속 관찰, 주기적 현장 순찰 등.  
- 기록·분석: 주요 수치·유지보수 상황을 전자파일로 기록(상황 변동 시마다, 주요 수치는 연구원이 정한 간격으로 주기 기록). 연구원 요청 시 분석 수행(대상·방법·결과물 형식은 연구원이 결정).  
- 장비·요건: 연구원은 제어용 컴퓨터·모니터·키보드·마우스·책상·의자 제공. 계약자는 기록·분석용 노트북·키보드·마우스 등 준비 및 연구원이 제공하는 제어시스템 교육 이해를 위한 배경지식 보유 필요.  
- 통신/보고: 현장 확인 필요 상황 발생 시 연구원이 무전(무전기 제공)으로 통신·지시. 일부 관찰은 비방사선구역에 한정.

모니터링 업무(문서에 명시된 항목 — 표 9 / 표 4 기반)
| 순번 | 모니터링 업무 | 비고 / 적용기간 |
|---:|---|---|
| 1 | 알람 메시지 확인 | 휴지기간·가동기간 공통 |
| 2 | 주요 센서 트렌드 확인

## 10. 비용 분석

In [18]:
# gpt-5-mini 가격 기준 (추정)
# input: $0.15/1M tokens, output: $0.60/1M tokens
est_input_price = 0.15 / 1_000_000
est_output_price = 0.60 / 1_000_000

print("=== 질문 1건당 비용 추정 ===")
print(f"  평균 입력 토큰: ~3,000 (시스템+컨텍스트+질문)")
print(f"  평균 출력 토큰: ~500")
print(f"  예상 비용: ${3000*est_input_price + 500*est_output_price:.4f} (약 {(3000*est_input_price + 500*est_output_price)*1350:.1f}원)")
print(f"\n  100건 질문: 약 {(3000*est_input_price + 500*est_output_price)*100*1350:.0f}원")
print(f"  1000건 질문: 약 {(3000*est_input_price + 500*est_output_price)*1000*1350:.0f}원")

=== 질문 1건당 비용 추정 ===
  평균 입력 토큰: ~3,000 (시스템+컨텍스트+질문)
  평균 출력 토큰: ~500
  예상 비용: $0.0008 (약 1.0원)

  100건 질문: 약 101원
  1000건 질문: 약 1012원


## 12. Enhanced RAG 파이프라인

5가지 검색 고도화를 적용한 버전입니다.
1. **섹션 필터**: 질문 의도에 맞는 문서 섹션 우선 검색
2. **content_type 부스팅**: 정형 데이터 질문에 표 청크 우선
3. **사업 요약 2단계 검색**: 요약으로 관련 문서 특정 후 청크 검색
4. **인접 청크 확장**: 관련 청크 앞뒤 청크를 컨텍스트에 추가
5. **금액/연도 범위 필터**: 수치 조건 자동 추출

In [19]:
# 사업 요약 임베딩 (문서 단위 1차 필터링용)
import numpy as np

summary_df = chunks_df.groupby('파일명').first()[['사업명', '발주 기관']].reset_index()
# CSV에서 사업 요약 로딩
meta_df = pd.read_parquet(Path('../../data/processed/cleaned_documents.parquet'))
summary_df = summary_df.merge(
    meta_df[['파일명', '사업 요약']].drop_duplicates(),
    on='파일명', how='left',
)
summary_df['요약_텍스트'] = summary_df.apply(
    lambda r: f"[{r['발주 기관']}] {r['사업명']}: {r['사업 요약']}", axis=1
)

# 100개 문서 요약 임베딩
summary_texts = summary_df['요약_텍스트'].tolist()
summary_response = openai_client.embeddings.create(input=summary_texts, model=EMBEDDING_MODEL)
summary_embeddings = np.array([d.embedding for d in summary_response.data])

print(f"사업 요약 임베딩: {len(summary_embeddings)}개 문서, 차원 {summary_embeddings.shape[1]}")


사업 요약 임베딩: 100개 문서, 차원 1536


In [20]:
# 섹션 키워드 매핑
SECTION_KEYWORDS = {
    '사업개요': ['목적', '배경', '왜', '개요', '일반', '기간', '규모'],
    '요구사항': ['요구사항', '기능', '성능', '보안', '요건', '조건'],
    '평가': ['평가', '배점', '선정', '심사', '기준'],
    '예산': ['예산', '금액', '비용', '산출', '단가', '대금'],
    '일정': ['일정', '기간', '착수', '완료', '마감'],
    '보안': ['보안', '개인정보', '암호', '접근통제'],
    '인력': ['인력', '투입', 'PM', '기술자', '조직'],
}


def extract_section_hint(query: str) -> str:
    """질문에서 관련 섹션 키워드를 추출한다."""
    for section, keywords in SECTION_KEYWORDS.items():
        if any(kw in query for kw in keywords):
            return section
    return None


def extract_range_filters(query: str) -> dict:
    """금액/연도 범위 필터를 추출한다."""
    import re
    where = {}
    
    # 금액: 'N억 이상', 'N억 이하'
    m = re.search(r'(\d+)억\s*이상', query)
    if m:
        where['사업금액'] = {'$gte': int(m.group(1)) * 100_000_000}
    m = re.search(r'(\d+)억\s*이하', query)
    if m:
        where['사업금액'] = {'$lte': int(m.group(1)) * 100_000_000}
    
    # 연도: '2024년'
    m = re.search(r'(202[0-9])년', query)
    if m:
        where['공개연도'] = int(m.group(1))
    
    return where if where else None


def find_relevant_docs(query: str, top_n: int = 3) -> list[str]:
    """사업 요약 임베딩으로 관련 문서 Top-N을 찾는다."""
    query_emb = np.array(embed_query(query))
    # 코사인 유사도
    sims = summary_embeddings @ query_emb / (
        np.linalg.norm(summary_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_idx = np.argsort(sims)[::-1][:top_n]
    return [summary_df.iloc[i]['파일명'] for i in top_idx]


def retrieve_enhanced(
    query: str,
    k: int = 5,
    where: dict = None,
    section_hint: str = None,
    boost_tables: bool = False,
    use_summary: bool = True,
) -> list[dict]:
    """Enhanced 검색: 사업요약 2단계 + 섹션 필터 + 표 부스팅."""
    query_embedding = embed_query(query)
    
    # 사업 요약으로 문서 범위 좁히기
    if use_summary and where is None:
        relevant_docs = find_relevant_docs(query, top_n=3)
        where = {'파일명': {'$in': relevant_docs}}
    
    # 기본 검색
    kwargs = {'query_embeddings': [query_embedding], 'n_results': k}
    if where:
        kwargs['where'] = where
    
    # 섹션 힌트가 있으면 문서 내용 필터
    if section_hint:
        kwargs['where_document'] = {'$contains': section_hint}
    
    results = collection.query(**kwargs)
    
    chunks = []
    for i in range(len(results['ids'][0])):
        score = 1 - results['distances'][0][i]
        # 표 부스팅: 표 청크의 점수를 1.2배
        if boost_tables and results['metadatas'][0][i].get('content_type') == 'table':
            score *= 1.2
        chunks.append({
            'score': round(score, 4),
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
        })
    
    # 부스팅 후 재정렬
    if boost_tables:
        chunks.sort(key=lambda c: c['score'], reverse=True)
    
    return chunks


def expand_neighbors(chunks: list[dict], n_neighbors: int = 1) -> list[dict]:
    """검색된 청크의 앞뒤 인접 청크를 추가한다."""
    expanded = list(chunks)
    seen_ids = set()
    
    for c in chunks:
        doc_name = c['metadata'].get('파일명', c['metadata'].get('사업명', ''))
        chunk_idx = c['metadata'].get('chunk_index', -1)
        if chunk_idx < 0:
            continue
        
        # 인접 청크 ID 생성
        for offset in range(-n_neighbors, n_neighbors + 1):
            if offset == 0:
                continue
            neighbor_idx = chunk_idx + offset
            if neighbor_idx < 0:
                continue
            neighbor_id = f"{doc_name}_{neighbor_idx}"
            if neighbor_id in seen_ids:
                continue
            seen_ids.add(neighbor_id)
            
            try:
                result = collection.get(ids=[neighbor_id], include=['documents', 'metadatas'])
                if result['documents']:
                    expanded.append({
                        'score': 0.0,  # 인접 청크는 점수 없음
                        'text': result['documents'][0],
                        'metadata': result['metadatas'][0],
                    })
            except:
                pass
    
    return expanded


print("Enhanced 함수 정의 완료")


Enhanced 함수 정의 완료


In [21]:
def rag_query_enhanced(
    query: str,
    chat_history: list = None,
    k: int = 5,
) -> dict:
    """Enhanced RAG: 5가지 검색 고도화 적용."""
    start = time.time()
    
    # 1) 메타데이터 필터 (발주기관/도메인/기관유형)
    where = extract_filters(query, chat_history)
    
    # 2) 금액/연도 범위 필터
    range_filter = extract_range_filters(query)
    if range_filter:
        where = {**(where or {}), **range_filter}
    
    # 3) 섹션 힌트
    section_hint = extract_section_hint(query)
    
    # 4) 표 부스팅 여부 판단
    table_keywords = ['요구사항', '정리', '목록', '예산', '금액', '일정', '배점', '기준']
    boost_tables = any(kw in query for kw in table_keywords)
    
    # 5) 사업 요약 2단계 (발주기관 필터가 없을 때만)
    use_summary = where is None
    
    # Enhanced 검색
    chunks = retrieve_enhanced(
        query, k=k, where=where,
        section_hint=section_hint,
        boost_tables=boost_tables,
        use_summary=use_summary,
    )
    
    # 인접 청크 확장
    chunks = expand_neighbors(chunks, n_neighbors=1)
    
    # 컨텍스트 조합
    context = build_context(chunks)
    
    # LLM 호출
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    if chat_history:
        for msg in chat_history[-4:]:
            messages.append({'role': 'user', 'content': msg['user']})
            messages.append({'role': 'assistant', 'content': msg['assistant']})
    messages.append({'role': 'user', 'content': f"""다음 RFP 문서 내용을 참고하여 질문에 답변해주세요.\n\n## 참고 문서\n{context}\n\n## 질문\n{query}"""})
    
    response = openai_client.chat.completions.create(
        model=LLM_MODEL, messages=messages, max_completion_tokens=16000,
    )
    answer = response.choices[0].message.content or '(응답 없음)'
    elapsed = time.time() - start
    
    return {
        'query': query, 'answer': answer,
        'filter': where, 'section_hint': section_hint,
        'boost_tables': boost_tables, 'use_summary': use_summary,
        'chunks': chunks, 'context_chars': len(context),
        'elapsed_sec': round(elapsed, 1),
        'tokens': {
            'prompt': response.usage.prompt_tokens,
            'completion': response.usage.completion_tokens,
            'total': response.usage.total_tokens,
        },
    }


print("Enhanced RAG 파이프라인 정의 완료")


Enhanced RAG 파이프라인 정의 완료


## 13. Naive vs Enhanced 비교 테스트

In [22]:
test_queries = [
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘",
    "교육 관련 사업 찾아줘",
    "기초과학연구원 극저온시스템 사업에서 AI 기반 예측에 대한 요구사항이 있나?",
    "5억 이상 대규모 시스템 구축 사업 예산을 비교해줘",
]

for q in test_queries:
    print(f"\nQ: {q}")
    
    # Naive
    r_naive = rag_query(q)
    # Enhanced
    r_enhanced = rag_query_enhanced(q)
    
    print(f"  [Naive]    필터={r_naive['filter']} | {r_naive['context_chars']:,}자 | {r_naive['elapsed_sec']}초")
    print(f"  [Enhanced] 필터={r_enhanced['filter']} | 섹션={r_enhanced['section_hint']} | 표부스팅={r_enhanced['boost_tables']} | 요약={r_enhanced['use_summary']}")
    print(f"             {r_enhanced['context_chars']:,}자 | {r_enhanced['elapsed_sec']}초")
    print(f"  [Naive 답변 앞 100자]    {r_naive['answer'][:100]}")
    print(f"  [Enhanced 답변 앞 100자] {r_enhanced['answer'][:100]}")
    print("=" * 60)



Q: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
  [Naive]    필터={'발주기관': '국민연금공단'} | 4,865자 | 40.0초
  [Enhanced] 필터={'발주기관': '국민연금공단'} | 섹션=요구사항 | 표부스팅=True | 요약=False
             5,472자 | 68.6초
  [Naive 답변 앞 100자]    핵심 요약:
- 요구사항은 교육운영, 이러닝시스템 기능, 학사관리, 콘텐츠 개발·유지관리, 개인정보·정보보안, 프로젝트관리 등으로 분류되어 있으며, 고유번호(예: FUR-001, 
  [Enhanced 답변 앞 100자] 핵심(요약)
- 국민연금공단 2024년 이러닝시스템 운영 용역의 주요 요구사항은 교육운영, 이러닝시스템 기능·학사관리, 콘텐츠 개발·유지관리, 개인정보·정보보안, 프로젝트관리(인수

Q: 교육 관련 사업 찾아줘
  [Naive]    필터={'사업도메인': '교육/학습'} | 4,903자 | 26.2초
  [Enhanced] 필터={'사업도메인': '교육/학습'} | 섹션=None | 표부스팅=False | 요약=False
             4,903자 | 47.4초
  [Naive 답변 앞 100자]    핵심 요약
- 제공된 문서에서 다음 4개의 교육 관련 사업이 확인됩니다. 각 사업의 주요 확인 가능 항목을 표로 정리했습니다. (문서에 없는 정보는 명시함)

표: 확인된 교육 관
  [Enhanced 답변 앞 100자] 핵심 요약 — 제공된 문서에서 확인된 교육 관련 사업은 다음 4건입니다. 각 사업별 핵심을 먼저 제시하고, 문서에서 확인된 세부사항을 이어서 적습니다.

1) 지능정보화전략계획(I

Q: 기초과학연구원 극저온시스템 사업에서 AI 기반 예측에 대한 요구사항이 있나?
  [Naive]    필터={'발주기관': '기초과학연구원'} | 5,114자 | 14.5초
  [Enhanced] 필터={'발주기관': '기초과학연구원'} | 섹션=요구사항 | 표

## 14. 종합 요약 및 인사이트

### Generation 결과
- gpt-5-mini로 RAG 답변 생성 파이프라인 완성
- Naive baseline + Enhanced (5가지 고도화) 두 버전 구현
- Enhanced 비교 테스트에서 금액 범위 필터, 섹션 필터, 표 부스팅의 효과 확인

### Naive vs Enhanced 비교 결과

| 질문 유형 | Naive | Enhanced | 개선 |
|---|---|---|---|
| 발주기관 명시 | 발주기관 필터만 | + 섹션 필터 + 표 부스팅 | 컨텍스트 품질 향상 (4,865→5,472자) |
| 도메인 검색 | 도메인 필터 | 동일 (이미 적용) | 동일 |
| 수치 조건 | **필터 없음 → 1건** | **금액 필터 → 3건** | 핵심 차이 |
| 모호한 질의 | 벡터만 의존 | 사업 요약 2단계 | 문서 특정 정확도 향상 |

### 5가지 개선의 실제 작동 현황

| # | 개선 | 작동 조건 | 비교 테스트 결과 |
|---|---|---|---|
| 1 | 섹션 필터 | '요구사항', '평가', '예산' 등 키워드 감지 시 | 요구사항/예산 질문에서 관련 섹션 청크 우선 검색 |
| 2 | 표 부스팅 | '정리', '목록', '예산', '기준' 등 키워드 시 | 표 청크 점수 1.2배 → 정형 데이터 상위 노출 |
| 3 | 사업 요약 2단계 | 메타 필터가 없을 때만 | 필터가 있으면 스킵 (정상 동작) |
| 4 | 인접 청크 확장 | 항상 | Top-k 결과의 앞뒤 청크로 컨텍스트 보완 |
| 5 | 금액/연도 필터 | 'N억 이상/이하', '202X년' 감지 시 | **5억 이상 질문: 1건→3건 (가장 극적)** |

### 왜 이렇게 구현했는가

**사업 요약 2단계 검색**: 14,000개 청크에서 바로 검색하면 표 구조 유사도에 의한 오답 발생.
100개 문서 요약으로 먼저 관련 문서를 좁히면 검색 범위가 1/30로 축소.
단, 발주기관/도메인 필터가 있으면 이미 범위가 좁혀져 있으므로 스킵.

**섹션 필터**: RFP 문서는 구조가 유사 (개요→요구사항→평가→입찰).
'평가기준 알려줘'에 사업개요 청크가 나오면 무관한 컨텍스트.
질문 키워드로 필요한 섹션을 추론하여 검색 범위를 좁힘.

**표 부스팅**: RFP 핵심 정보(예산/요구사항/평가배점)는 표에 집중.
37%가 표 청크인데 가중치 없이 검색하면 텍스트 청크에 밀릴 수 있음.
정형 데이터 질문에서만 선택적으로 표 점수를 1.2배.

**인접 청크 확장**: 1,000자 청크 단위로 잘린 문맥을 앞뒤로 보완.
요구사항 표가 여러 청크에 걸쳐있을 때 뒷부분을 함께 가져옴.

**금액/연도 범위 필터**: '5억 이상 사업' 같은 수치 조건은 벡터 유사도로 처리 불가.
정규식으로 금액/연도를 추출하여 ChromaDB의 $gte/$lte 연산자로 필터링.
이전 Naive에서 1건만 찾던 질문이 Enhanced에서 3건으로 개선된 핵심 요인.

### 다음 단계
- `08_evaluation.ipynb`: Naive vs Enhanced 정량 비교 평가
- 시나리오 A (KURE-v1) 임베딩 비교 실험
